In [1]:
# loading the samanantar dataset 
from datasets import load_dataset
dataset = load_dataset("ai4bharat/samanantar", "hi", trust_remote_code=True)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ai4bharat/samanantar' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

hi/train-00000-of-00008.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

hi/train-00001-of-00008.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

hi/train-00002-of-00008.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

...

hi/train-00004-of-00008.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

hi/train-00005-of-00008.parquet:   0%|          | 0.00/239M [00:00<?, ?B/s]

hi/train-00006-of-00008.parquet:   0%|          | 0.00/239M [00:00<?, ?B/s]

hi/train-00007-of-00008.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10125706 [00:00<?, ? examples/s]

In [2]:
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['idx', 'src', 'tgt'],
        num_rows: 10125706
    })
})
{'idx': 0, 'src': "However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles", 'tgt': 'आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।'}


In [ ]:
# filtering the entire dataset

import re

def is_good_pair(example):
    src = example['src'].strip()
    tgt = example['tgt'].strip()

    if not src or not tgt: # removing enpty pair
        return False

    # Sentence length filter 
    src_len = len(src.split())
    tgt_len = len(tgt.split())
    if src_len < 3 or src_len > 50:
        return False
    if tgt_len < 3 or tgt_len > 60:
        return False

    # Length ratio check
    ratio = src_len / tgt_len
    if ratio < 0.4 or ratio > 2.5:
        return False

    # Hindi must contain devanagari script
    devanagari = re.compile(r'[\u0900-\u097F]')
    if not devanagari.search(tgt):
        return False

    # English side shouldnt contain devanagari
    if devanagari.search(src):
        return False

    return True

print("Filtering... this will take 2-3 minutes")
filtered = dataset['train'].filter(is_good_pair, num_proc=4)
print(f"Before: {len(dataset['train']):,}")
print(f"After filtering: {len(filtered):,}")

Filtering... this will take 2-3 minutes


Filter (num_proc=4):   0%|          | 0/10125706 [00:00<?, ? examples/s]

Before: 10,125,706
After filtering: 9,638,933


In [5]:
# sampling 150k datapoints randomly
sampled = filtered.shuffle(seed=42).select(range(150_000))
print(f"Sampled dataset size: {len(sampled):,}")
print("\nSample check:")
print("English:", sampled[0]['src'])
print("Hindi:  ", sampled[0]['tgt'])

Sampled dataset size: 150,000

Sample check:
English: Shri M.P Chaudhari, Director (Finance) and other senior officials of MOIL Limited were also present.
Hindi:   चौधरी, निदेशक (वित्त) और मॉयल लिमिटेड के अन्य वरिष्ठ अधिकारीगण भी इस अवसर पर उपस्थित थे।


In [6]:
# train val test split 
train_data = sampled.select(range(0, 120_000))
val_data   = sampled.select(range(120_000, 135_000))
test_data  = sampled.select(range(135_000, 150_000))

print(f"Train : {len(train_data):,}")
print(f"Val   : {len(val_data):,}")
print(f"Test  : {len(test_data):,}")

Train : 120,000
Val   : 15,000
Test  : 15,000


In [7]:
# saving the splitted datase t

import pandas as pd

def save_split(split, filename):
    df = pd.DataFrame({
        'english': [ex['src'] for ex in split],
        'hindi':   [ex['tgt'] for ex in split]
    })
    df.to_csv(filename, index=False, encoding='utf-8')
    print(f"Saved {filename} — {len(df):,} rows")

save_split(train_data, 'train.csv')
save_split(val_data,   'val.csv')
save_split(test_data,  'test.csv')

Saved train.csv — 120,000 rows
Saved val.csv — 15,000 rows
Saved test.csv — 15,000 rows


In [8]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
validate = pd.read_csv('val.csv')
print(train.head(3))
print("\nAny nulls?", train.isnull().sum().to_dict())

                                             english                                              hindi
0  Shri M.P Chaudhari, Director (Finance) and oth...  चौधरी, निदेशक (वित्त) और मॉयल लिमिटेड के अन्य ...
1  The court has already granted them interim rel...  कोर्ट ने फिलहाल उन्हें गिरफ्तारी से अंतरिम संर...
2              This is the biggest advantage for us.               सबसे बड़ा फायदा हमारे लिए तो यही है.

Any nulls? {'english': 0, 'hindi': 0}


In [9]:
print(test.head(3))
print("\nAny nulls?", test.isnull().sum().to_dict())

                                             english                                              hindi
0  A Pakistani court has constituted a two-member...  पाकिस्तान की एक अदालत ने अल-अज़ी़जि़या भ्रष्टा...
1            The explosion caused panic in the area.      इस विस्फोट के चलते क्षेत्र में हड़कंप मच गया।
2                           Government approved this                                 सरकार ने दी मंजूरी

Any nulls? {'english': 0, 'hindi': 0}


In [10]:
print(validate.head(3))
print("\nAny nulls?", validate.isnull().sum().to_dict())

                                             english                                              hindi
0       I did not come to politics after retirement.  मैं संन्यास लेने के लिए राजनीति में नहीं आया हूं.
1  There were numerous attempts to give the murde...  हत्या को राजनीतिक लाभ के लिए एक सांप्रदायिक को...
2  It was explained that the NSS had been conduct...  यह बताया गया था कि एनएसएस यह सर्वे सेवा क्षेत्...

Any nulls? {'english': 0, 'hindi': 0}


In [11]:
train.shape , validate.shape , test.shape

((120000, 2), (15000, 2), (15000, 2))